In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# Функция для сохранения ответов в формате Coursera (без \n на конце)
def save_answer(filename, answer_str):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(str(answer_str).strip())
    print(f"Ответ для '{filename}' успешно сохранен: {answer_str}")

# 1. Загрузка данных
prices_df = pd.read_csv('close_prices.csv')
djia_df = pd.read_csv('djia_index.csv')

# Выделяем только числовые признаки (убираем колонку с датами)
X = prices_df.drop(columns=['date'])

# 2. Обучение PCA с 10 компонентами
pca = PCA(n_components=10)
pca.fit(X)

# Считаем, сколько компонент нужно для объяснения >= 90% дисперсии
# pca.explained_variance_ratio_ содержит долю дисперсии для каждой компоненты
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
components_count = np.argmax(cumulative_variance >= 0.90) + 1

save_answer('pca_ans1.txt', components_count)

# 3. Применение трансформации и получение первой главной компоненты
X_transformed = pca.transform(X)
first_comp = X_transformed[:, 0]

# 4. Расчет корреляции Пирсона между первой компонентой и индексом Доу-Джонса
# Индекс DJI находится во втором столбце файла djia_index.csv
dji_index = djia_df.iloc[:, 1].values

# np.corrcoef возвращает матрицу корреляции 2x2, берем элемент [0, 1]
corr_matrix = np.corrcoef(first_comp, dji_index)
pearson_corr = corr_matrix[0, 1]

save_answer('pca_ans2.txt', f"{pearson_corr:.2f}")

# 5. Поиск компании с наибольшим весом в первой главной компоненте
# Веса (коэффициенты линейной комбинации) лежат в pca.components_[0]
first_comp_weights = pca.components_[0]

# Находим индекс максимального веса
best_company_idx = np.argmax(first_comp_weights)
# Находим имя компании по названию столбца
best_company_name = X.columns[best_company_idx]

save_answer('pca_ans3.txt', best_company_name)

# --- Контрольный вывод для проверки ---
print("\n--- Проверка результатов ---")
print(f"1. Количество компонент для 90% дисперсии: {components_count}")
print(f"2. Корреляция Пирсона с индексом Доу-Джонса: {pearson_corr:.4f}")
print(f"3. Компания с наибольшим весом: {best_company_name} (вес: {first_comp_weights[best_company_idx]:.4f})")

Ответ для 'pca_ans1.txt' успешно сохранен: 4
Ответ для 'pca_ans2.txt' успешно сохранен: 0.91
Ответ для 'pca_ans3.txt' успешно сохранен: V

--- Проверка результатов ---
1. Количество компонент для 90% дисперсии: 4
2. Корреляция Пирсона с индексом Доу-Джонса: 0.9097
3. Компания с наибольшим весом: V (вес: 0.5797)
